# 17 · DiT conditioning (adaLN-Zero), flow matching, classifier-free guidance

**Needed for:** DiT (vision track, step V4). Flow matching is what current image and video models (Stable
Diffusion 3, Flux) and Janus-Pro's successors train with.

Three ideas, each small:
1. **adaLN-Zero (DiT):** condition a transformer block on timestep + class by predicting LayerNorm shift/scale and
   a residual gate from the conditioning vector. Initialize the predictor to zero so every block starts as the
   identity.
2. **Flow matching / rectified flow:** draw a straight line from data $x_0$ to noise $\epsilon$,
   $x_t = (1-t)\,x_0 + t\,\epsilon$, and train the network to predict the velocity $v = \epsilon - x_0$.
   Sample by integrating the velocity from $t=1$ back to $t=0$ with a few Euler steps.
3. **Classifier-free guidance (CFG):** train with the class label dropped 10% of the time, then at sampling push
   the prediction away from the unconditional one: $v = v_u + w\,(v_c - v_u)$.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"


def drill(test, fn):
    """Run a test against your own implementation; prints TODO until you write it."""
    try:
        test(fn)
    except NotImplementedError:
        print(f"TODO  {fn.__name__}")
    else:
        print(f"PASS  {fn.__name__}")

## 1. adaLN-Zero

From the conditioning vector $c$ (timestep embedding + class embedding), a linear layer predicts six vectors:
shift, scale and gate for the attention branch and the same for the MLP branch.
$$h = x + g_1 \odot \text{Attn}\big(\text{LN}(x)(1 + s_1) + b_1\big), \qquad \text{out} = h + g_2 \odot \text{MLP}\big(\text{LN}(h)(1 + s_2) + b_2\big)$$
With the predictor's weights at zero, $g = 0$ and the block returns $x$ unchanged. DiT found this stabilizes
training noticeably compared with other conditioning methods.

In [ ]:
def modulate(x, shift, scale):
    """x: (B, N, D); shift, scale: (B, D)."""
    return x * (1 + scale[:, None]) + shift[:, None]


class DiTBlock(nn.Module):
    def __init__(self, dim, heads=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.mlp = nn.Sequential(nn.Linear(dim, 4 * dim), nn.GELU(approximate="tanh"), nn.Linear(4 * dim, dim))
        self.ada = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        nn.init.zeros_(self.ada[1].weight)
        nn.init.zeros_(self.ada[1].bias)

    def forward(self, x, c):
        s1, sc1, g1, s2, sc2, g2 = self.ada(c).chunk(6, dim=-1)
        h = modulate(self.norm1(x), s1, sc1)
        x = x + g1[:, None] * self.attn(h, h, h, need_weights=False)[0]
        return x + g2[:, None] * self.mlp(modulate(self.norm2(x), s2, sc2))


def test_adaln_zero(block_cls):
    torch.manual_seed(0)
    blk = block_cls(32)
    x, c = torch.randn(2, 5, 32), torch.randn(2, 32)
    torch.testing.assert_close(blk(x, c), x)                       # identity at init
    with torch.no_grad():
        blk.ada[1].weight.normal_(std=0.1)
    out1 = blk(x, c)
    assert not torch.allclose(out1, x)
    assert not torch.allclose(out1, blk(x, c + 1.0))               # output depends on the conditioning


test_adaln_zero(DiTBlock)
print("adaLN-Zero block is the identity at init and conditioned after")

## 2. Flow matching loss and Euler sampling

Time runs from $t=0$ (data) to $t=1$ (noise). Sample $t$ uniformly (SD3 uses a logit-normal distribution that
favours middle times). The velocity target is constant along each straight path, which is why few sampling steps
can work.

In [ ]:
def flow_matching_loss(model, x0, cond):
    t = torch.rand(x0.shape[0], device=x0.device)
    noise = torch.randn_like(x0)
    x_t = (1 - t[:, None]) * x0 + t[:, None] * noise
    return F.mse_loss(model(x_t, t, cond), noise - x0)


@torch.no_grad()
def euler_sample(velocity_fn, noise, steps):
    """Integrate dx/dt = v from t=1 (noise) to t=0 (data)."""
    x = noise
    ts = torch.linspace(1.0, 0.0, steps + 1, device=noise.device)
    for i in range(steps):
        t, t_next = ts[i], ts[i + 1]
        x = x + (t_next - t) * velocity_fn(x, t.expand(x.shape[0]))
    return x


def test_euler(fn):
    # with the exact velocity of a single straight path, Euler lands exactly on x0 in any number of steps
    x0, eps = torch.randn(3, 2), torch.randn(3, 2)
    exact_v = lambda x, t: eps - x0
    for steps in (1, 4, 10):
        torch.testing.assert_close(fn(exact_v, eps, steps), x0)


test_euler(euler_sample)
print("Euler integration recovers x0 from the exact velocity")

## 3. Class-conditional toy model with CFG

Same 8-Gaussian ring as notebook 16, but each mode is wide (std 0.6) so neighbouring classes overlap, and the model
is conditioned on which of the 8 modes to generate. Label 8 means "no label"
(used for the dropped 10% during training and for the unconditional branch at sampling). The network conditions
through an adaLN-style shift/scale from (time, label), like DiT, but on an MLP instead of attention.

In [ ]:
CENTERS = torch.stack([torch.tensor([math.cos(2 * math.pi * i / 8), math.sin(2 * math.pi * i / 8)]) * 2 for i in range(8)])
NULL = 8
SPREAD = 0.6


def timestep_embedding(t, dim=64):
    freqs = torch.exp(-math.log(1000) * torch.arange(dim // 2, device=t.device) / (dim // 2))
    args = t[:, None] * 1000 * freqs[None]
    return torch.cat([args.cos(), args.sin()], -1)


class VelocityNet(nn.Module):
    def __init__(self, dim=128):
        super().__init__()
        self.label = nn.Embedding(9, dim)
        self.t_mlp = nn.Sequential(nn.Linear(64, dim), nn.SiLU(), nn.Linear(dim, dim))
        self.inp = nn.Linear(2, dim)
        self.layers = nn.ModuleList(nn.Linear(dim, dim) for _ in range(3))
        self.ada = nn.ModuleList(nn.Linear(dim, 2 * dim) for _ in range(3))
        self.out = nn.Linear(dim, 2)

    def forward(self, x, t, y):
        c = F.silu(self.t_mlp(timestep_embedding(t)) + self.label(y))
        h = self.inp(x)
        for lin, ada in zip(self.layers, self.ada):
            shift, scale = ada(c).chunk(2, -1)
            h = h + lin(F.silu(F.layer_norm(h, h.shape[-1:]) * (1 + scale) + shift))
        return self.out(h)


torch.manual_seed(0)
net = VelocityNet().to(device)
opt = torch.optim.AdamW(net.parameters(), lr=2e-3)
for step in range(3001):
    y = torch.randint(0, 8, (512,), device=device)
    x0 = CENTERS.to(device)[y] + SPREAD * torch.randn(512, 2, device=device)
    y_train = torch.where(torch.rand(512, device=device) < 0.1, torch.full_like(y, NULL), y)   # label dropout
    loss = flow_matching_loss(net, x0, y_train)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 1000 == 0:
        print(f"step {step:4d}  loss {loss.item():.4f}")

In [ ]:
def cfg_velocity(net, y, w):
    def v(x, t):
        v_c = net(x, t, y)
        v_u = net(x, t, torch.full_like(y, NULL))
        return v_u + w * (v_c - v_u)
    return v


def class_accuracy(x, y):
    return (torch.cdist(x.cpu(), CENTERS).argmin(1) == y.cpu()).float().mean().item()


n = 4000
y = torch.randint(0, 8, (n,), device=device)
real = CENTERS[y.cpu()] + SPREAD * torch.randn(n, 2)
print(f"real data: nearest mode is the labelled one for {class_accuracy(real, y):.0%}, "
      f"median distance to it {(real - CENTERS[y.cpu()]).norm(dim=1).median():.2f}\n")
noise = torch.randn(n, 2, device=device)
for w in (0.0, 1.0, 2.0, 4.0):
    x = euler_sample(cfg_velocity(net, y, w), noise, 32)
    spread = (x.cpu() - CENTERS[y.cpu()]).norm(dim=1).median().item()
    print(f"guidance w={w:.0f}: nearest mode is the requested one for {class_accuracy(x, y):.0%}, "
          f"median distance to it {spread:.2f}")
print()
for steps in (1, 4, 32):
    x = euler_sample(cfg_velocity(net, y, 1.0), noise, steps)
    spread = (x.cpu() - CENTERS[y.cpu()]).norm(dim=1).median().item()
    print(f"w=1 with {steps:2d} Euler steps: {class_accuracy(x, y):.0%} on the requested mode, median distance {spread:.2f}")

Compare every row with the real data's numbers (81% and 0.69 on a typical run):
- $w = 0$ ignores the label, so about 1 in 8 samples land on the requested mode by chance.
- $w = 1$ is the plain conditional model and matches the real data: the classes overlap, so some samples
  correctly sit nearer a neighbouring mode.
- $w = 2$ over-commits to the class: every sample lands on the requested mode. That's "more on-class, less
  diverse", the look of high-guidance images.
- $w = 4$ overshoots: samples are pushed *past* the mode, away from the other classes, and distance grows again.
  In image models this is the saturated, over-contrasted look of too much guidance.
- **Steps:** one Euler step lands every sample on the class *average*, which scores 100% but has no diversity at
  all. With 32 steps the distribution matches the real data. Accuracy alone would pick the wrong setting here;
  always check spread as well.

## Things to be able to explain

- How does CFG double the cost of sampling, and how do distilled models avoid it?
- Why do straight paths (rectified flow) allow fewer sampling steps than DDPM's curved ones?
- In a real DiT, the input is a VAE latent (e.g. 32×32×4), patchified exactly like notebook 13, and the output
  is unpatchified back to the latent shape.

## Drill

In [ ]:
def my_modulate(x, shift, scale):
    raise NotImplementedError


def my_euler_sample(velocity_fn, noise, steps):
    raise NotImplementedError


def test_modulate(fn):
    x, shift, scale = torch.randn(2, 3, 4), torch.randn(2, 4), torch.randn(2, 4)
    torch.testing.assert_close(fn(x, shift, scale), modulate(x, shift, scale))
    torch.testing.assert_close(fn(x, torch.zeros(2, 4), torch.zeros(2, 4)), x)


drill(test_modulate, my_modulate)
drill(test_euler, my_euler_sample)